In [1]:
import numpy as np

latent_vectors_long = np.load(
    "../extract_latents/latent_vectors_len60_v2.npy"
)

encoded_sequences = np.load(
    "../encoded_sequences_len60_v2.npy"
)

print(latent_vectors_long.shape)
print(encoded_sequences.shape)

(995, 64)
(995, 62)


In [2]:
#Train/Test Split
from sklearn.model_selection import train_test_split

X_train_latent, X_test_latent, y_train_seq, y_test_seq = train_test_split(
    latent_vectors_long,
    encoded_sequences,
    test_size=0.1,
    random_state=42
)

print(X_train_latent.shape)
print(X_test_latent.shape)

print(y_train_seq.shape)
print(y_test_seq.shape)

(895, 64)
(100, 64)
(895, 62)
(100, 62)


In [3]:
#Convert to Tensor
import torch

X_train_latent = torch.tensor(
    X_train_latent,
    dtype=torch.float32
)

X_test_latent = torch.tensor(
    X_test_latent,
    dtype=torch.float32
)

y_train_seq = torch.tensor(
    y_train_seq,
    dtype=torch.long
)

y_test_seq = torch.tensor(
    y_test_seq,
    dtype=torch.long
)

print(X_train_latent.shape)
print(y_train_seq.shape)

torch.Size([895, 64])
torch.Size([895, 62])


In [4]:
#Create DataLoader
from torch.utils.data import (
    TensorDataset,
    DataLoader
)

train_loader = DataLoader(
    TensorDataset(
        X_train_latent,
        y_train_seq
    ),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(
        X_test_latent,
        y_test_seq
    ),
    batch_size=32,
    shuffle=False
)

print(len(train_loader))

28


In [5]:
print(X_train_latent.shape)
print(y_train_seq.shape)
print(len(train_loader))

torch.Size([895, 64])
torch.Size([895, 62])
28


In [6]:
#LSTMDecoder
import torch
import torch.nn as nn

class LongPeptideLSTMDecoder(
    nn.Module
):

    def __init__(
        self,
        latent_dim=64,
        hidden_dim=256,
        vocab_size=23,
        max_len=62
    ):

        super().__init__()

        self.max_len = max_len

        self.latent_to_hidden = nn.Linear(
            latent_dim,
            hidden_dim
        )

        self.embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=2,
            dropout=0.3,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        latent,
        seq
    ):

        hidden = torch.tanh(
            self.latent_to_hidden(
                latent
            )
        )

        h0 = hidden.unsqueeze(0).repeat(
            2,
            1,
            1
        )

        c0 = torch.zeros_like(
            h0
        )

        emb = self.embedding(
            seq[:,:-1]
        )

        out, _ = self.lstm(
            emb,
            (h0,c0)
        )

        logits = self.fc(
            out
        )

        return logits

In [7]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

decoder = LongPeptideLSTMDecoder().to(
    device
)

print(
    sum(
        p.numel()
        for p in decoder.parameters()
    )
)

947095


In [8]:
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)

In [9]:
optimizer = torch.optim.Adam(
    decoder.parameters(),
    lr=5e-4
)

In [10]:
num_epochs = 150

for epoch in range(num_epochs):

    decoder.train()

    total_loss = 0

    for latent, seq in train_loader:

        latent = latent.to(device)

        seq = seq.to(device)

        logits = decoder(
            latent,
            seq
        )

        loss = criterion(
            logits.reshape(
                -1,
                23
            ),
            seq[:,1:].reshape(-1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss={total_loss/len(train_loader):.4f}"
    )

Epoch [1/150] Loss=2.9975
Epoch [2/150] Loss=2.8804
Epoch [3/150] Loss=2.8296
Epoch [4/150] Loss=2.7825
Epoch [5/150] Loss=2.7380
Epoch [6/150] Loss=2.6938
Epoch [7/150] Loss=2.6475
Epoch [8/150] Loss=2.6006
Epoch [9/150] Loss=2.5555
Epoch [10/150] Loss=2.5129
Epoch [11/150] Loss=2.4692
Epoch [12/150] Loss=2.4242
Epoch [13/150] Loss=2.3857
Epoch [14/150] Loss=2.3394
Epoch [15/150] Loss=2.2925
Epoch [16/150] Loss=2.2567
Epoch [17/150] Loss=2.2123
Epoch [18/150] Loss=2.1740
Epoch [19/150] Loss=2.1297
Epoch [20/150] Loss=2.0948
Epoch [21/150] Loss=2.0548
Epoch [22/150] Loss=2.0161
Epoch [23/150] Loss=1.9750
Epoch [24/150] Loss=1.9378
Epoch [25/150] Loss=1.9117
Epoch [26/150] Loss=1.8714
Epoch [27/150] Loss=1.8328
Epoch [28/150] Loss=1.8031
Epoch [29/150] Loss=1.7714
Epoch [30/150] Loss=1.7411
Epoch [31/150] Loss=1.7122
Epoch [32/150] Loss=1.6753
Epoch [33/150] Loss=1.6422
Epoch [34/150] Loss=1.6157
Epoch [35/150] Loss=1.5856
Epoch [36/150] Loss=1.5564
Epoch [37/150] Loss=1.5306
Epoch [38/

In [11]:
decoder.eval()

correct = 0
total = 0

with torch.no_grad():

    for latent, seq in test_loader:

        latent = latent.to(device)

        seq = seq.to(device)

        logits = decoder(
            latent,
            seq
        )

        pred = logits.argmax(
            dim=2
        )

        target = seq[:,1:]

        mask = (
            target != 0
        )

        correct += (
            ((pred == target) & mask)
            .sum()
            .item()
        )

        total += (
            mask.sum().item()
        )

token_acc = correct / total

print(
    f"Token Accuracy: {token_acc:.4f}"
)

Token Accuracy: 0.4457


In [12]:
decoder.eval()

with torch.no_grad():

    latent = X_test_latent[:5].to(device)

    seq = y_test_seq[:5].to(device)

    logits = decoder(
        latent,
        seq
    )

    pred = logits.argmax(dim=2)

for i in range(5):

    print("="*50)

    print("TARGET")
    print(
        seq[i,1:40]
        .cpu()
        .numpy()
    )

    print("PRED")
    print(
        pred[i,:39]
        .cpu()
        .numpy()
    )

TARGET
[17 10 16 16 10  6 16 11 10  9  9 10  6 16 17 10 16 16 10  6 16 17 10 18
  8 10 20 16 16 16 14 14 12 12 17  3 10  6  3]
PRED
[17 16 16 10 10 10 16 11 10  9  9 10  6 16 17 10 16 16 10  6 16 17  3  6
  6  6  6 16 12 12 12  3  3  3  3 18 19  8  8]
TARGET
[15 20 18 17 16 16  4 18 16 17 10 16  8  6 17  7 14 16  4 17 18 16 13 16
  5  8 16 12 16 18  4  4 16  6 12 16 14 20  6]
PRED
[17 16 16 16  3  3  4 22 22 20  3  8  8 20 20  9  9  9 20 17 20 12 12 15
  8 17 17 10 17 17 17 17 17 17 17 17 17  4 17]
TARGET
[16  8 22 11  8 15 22 19 17 15 10 12 17 15 22 20 17 15 20 20 18 22 14 20
  4 19 12 18  4 17  8 10 19 19 19 16  3 17 18]
PRED
[16  8 22 11  8 15 22 19 17 15 10 12 17 15 22 20 17 15 20 20 18 22 14  3
  3 19 12 18  4 17 10 10 19 19 16 16  3 17  4]
TARGET
[ 3 19  4  5 12  3 18  7 18 18 16 21 20 19 15 14  5 18 12  4  3  3  9  4
 12 20 11  8 22 17  8  8 22  4 11 14 11 10  4]
PRED
[11 19  4  5 12 12 18  7 18 18 16 21 20 19 15 14  5 18 12  4  3  3  9  4
 10  3  3 17 17 17  8  8 17  4 11 14 11

In [13]:
torch.save(
    decoder.state_dict(),
    "lstm_decoder_len60.pt"
)

In [14]:
torch.save(
    {
        "model_state_dict":
            decoder.state_dict()
    },
    "lstm_decoder_checkpoint_len60.pt"
)

In [15]:
AA_VOCAB = "ACDEFGHIKLMNPQRSTVWY"

idx_to_aa = {
    i+3: aa
    for i, aa in enumerate(
        AA_VOCAB
    )
}

START_TOKEN = 1
END_TOKEN = 2
PAD_TOKEN = 0

In [16]:
def generate_sequence(
    latent,
    max_len=62
):

    decoder.eval()

    latent = latent.to(device)

    hidden = torch.tanh(
        decoder.latent_to_hidden(
            latent
        )
    )

    h = hidden.unsqueeze(0).repeat(
        2,
        1,
        1
    )

    c = torch.zeros_like(h)

    token = torch.tensor(
        [[START_TOKEN]],
        device=device
    )

    generated = []

    for _ in range(max_len):

        emb = decoder.embedding(
            token
        )

        out, (h,c) = decoder.lstm(
            emb,
            (h,c)
        )

        logits = decoder.fc(
            out[:,-1]
        )

        temperature = 0.8
        probs = torch.softmax(
            logits / temperature,
            dim=-1
        )
        
        next_token = torch.multinomial(
            probs,
            num_samples=1
        ).squeeze(-1)

        token = next_token.unsqueeze(1)

        value = next_token.item()

        if value == END_TOKEN:
            break

        generated.append(value)

    return generated

In [17]:
import numpy as np

generated_latents = np.load(
    "generated_latents_len60_v2.npy"
)

print(generated_latents.shape)

(5000, 64)


In [18]:
z = torch.tensor(
    generated_latents[0:1],
    dtype=torch.float32
).to(device)

tokens = generate_sequence(z)

print(tokens)
print(len(tokens))

[17, 11, 17, 14, 19, 6, 7, 18, 15, 16, 11, 6, 3, 16, 11, 14, 14, 3, 18, 9, 3, 17, 11, 18, 22, 11, 12, 17, 7, 20, 18, 8, 8, 4, 11, 22, 10, 8, 14, 4, 17, 9]
42


In [19]:
peptide = "".join(
    [
        idx_to_aa[t]
        for t in tokens
        if t >= 3
    ]
)

print(peptide)
print(len(peptide))

RKRNTEFSPQKEAQKNNASHARKSYKLRFVSGGCKYIGNCRH
42


In [20]:
all_peptides = []

for i in range(5000):

    z = torch.tensor(
        generated_latents[i:i+1],
        dtype=torch.float32
    ).to(device)

    tokens = generate_sequence(z)

    peptide = "".join(
        [
            idx_to_aa[t]
            for t in tokens
            if t >= 3
        ]
    )

    if len(peptide) > 0:

        all_peptides.append(
            peptide
        )

    print("="*50)
    print(i)
    print(peptide)
    print(len(peptide))

0
RTCESQSHKFKGPCFSDSNCATVCRTEGFTRGDCNGHVRRCFCLRRC
47
1
QNLKAICEWLADTHGTCNSHCQKLCWRLEYKTGRCVKNPQRPCICLLPCTRFQNDLLNR
59
2
RVCRRSAGFKGKCVSDHNCAQVCLEEGYGGGNCDGIMRQCKCIRQC
46
3
GIPVNIIKKAVTCGNKGLCVRKNCELAMREKGQCKEVYMSCLCKITLSKE
50
4
FRPLQQPFRVPVRIASLRPACIPFGLQDTCSAFCRGFRQGAHCGTNLCKCRVAC
54
5
QKCGNPGHSQCSRFNCPSLGIRCKTGICSDKPPKLGKCCLKDGCS
45
6
ASYGGNEYCDNKSCWDVRGNEARCWKTHCKDIGSCQVYNACKIELPCRP
49
7
RECKTESNTFPGICITKPPCRKACISEKFTDGHCSKILRRCLCTKPC
47
8
DKCWILRGSCDVECSKDKCELAGFHGGCKEQLWGRPMECCCKYLTDG
47
9
NWYVKKCLNDVGICKKKCKPEEMHVKNGWAMCGKQRDCCVPAD
43
10
GKIPPQTYGQIVEGRCFTSRPYNCRVECLSGRRAGGYCNAEKGSLCCCVKY
51
11
RTCMSKKEGFKGCVDTTNCDHVCKSEGYPDGKCKGVRRCFCTKLC
45
12
NWYVKKCLNDVGICKKKCKPGEMHIKNGWATCGQKSDCCVPAD
43
13
QDRPKKPGLCPPRPQKPPCVRECKNDWSCPGEQKCCCRYGCIFECRDPIFVN
52
14
ATIGNGVSCNRRKCIVGRCRQEIIRCTTPQKCCKRKIEIGSKYD
44
15
RVCMEKSQHHFNICCPARFEDAEGCNKLMCRAKGHTGSCFYDIRESNQLCFCTYCN
56
16
RCLTASKKFGACVDTNCASVCHEGFPGGNCDGKLRRCYCTKHC
43
17
RMRCQQKLSNFHQGFCYTNRAHCIKQCKRWEMAGTCDQWVNCRCLIKECSPTRQNL
56
18
LGDK

In [21]:
print(
    len(all_peptides)
)

print(
    len(set(all_peptides))
)

print(
    len(set(all_peptides))
    /
    len(all_peptides)
)

5000
4861
0.9722


In [22]:
unique_peptides = list(
    set(all_peptides)
)

print(len(unique_peptides))

4861


In [23]:
import pandas as pd

unique_df = pd.DataFrame({
    "Sequence": unique_peptides
})

unique_df["Length"] = (
    unique_df["Sequence"].str.len()
)

unique_df.to_csv(
    "amp_len60.csv",
    index=False
)

filtered_df = unique_df[
    (unique_df["Length"] >= 41)
    &
    (unique_df["Length"] <= 60)
].copy()

print(filtered_df.shape)
print(filtered_df["Length"].min())
print(filtered_df["Length"].max())
print(filtered_df["Length"].mean())

(4436, 2)
41
60
48.20491433724076


In [24]:
filtered_df.to_csv(
    "long_peptides_filtered_41_60.csv",
    index=False
)

In [25]:
import torch
import esm

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

model = model.to(device)

model.eval()

batch_converter = (
    alphabet.get_batch_converter()
)

print("Loaded")

Loaded


In [26]:
import numpy as np
from tqdm import tqdm

sequences = filtered_df["Sequence"].tolist()

batch_size = 32

all_embeddings = []

for i in tqdm(
    range(
        0,
        len(sequences),
        batch_size
    )
):

    batch_seqs = (
        sequences[
            i:i+batch_size
        ]
    )

    batch = [
        (str(j), seq)
        for j, seq
        in enumerate(batch_seqs)
    ]

    _, _, tokens = (
        batch_converter(batch)
    )

    tokens = tokens.to(device)

    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[33],
            return_contacts=False
        )

    reps = results[
        "representations"
    ][33]

    for j, seq in enumerate(
        batch_seqs
    ):

        seq_len = len(seq)

        emb = (
            reps[
                j,
                1:seq_len+1
            ]
            .mean(0)
            .cpu()
            .numpy()
        )

        all_embeddings.append(
            emb
        )

generated_embeddings = np.array(
    all_embeddings
)

print(
    generated_embeddings.shape
)

100%|█████████████████████████████████████████████████████████████████████████████████| 139/139 [00:14<00:00,  9.27it/s]

(4436, 1280)


In [ ]:
import joblib

amp_model = joblib.load(
    "../../classifers/amp_xgb_classifier.pkl"
)

print(type(amp_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [28]:
amp_probs = amp_model.predict_proba(
    generated_embeddings
)[:,1]

print(
    amp_probs.shape
)

print(
    amp_probs.min()
)

print(
    amp_probs.max()
)

print(
    amp_probs.mean()
)

(4436,)
0.0023427668
0.99981695
0.7218379


In [29]:
print(len(all_peptides))
print(len(amp_probs))

5000
4436


In [30]:
import pandas as pd

results_df = pd.DataFrame(
    {
        "Sequence":  filtered_df["Sequence"].tolist(),
        "AMP_Probability": amp_probs
    }
)

results_df = results_df.sort_values(
    by="AMP_Probability",
    ascending=False
)

results_df.head()

,Sequence,AMP_Probability
1063,GIFSKLAGKIKNLFIKGAKNVGKRVGMDVVRTGIDVIGCKIKGEC,0.999817
2827,QKLCERSSGTWSGVCGNNNACKNQCINLEGARHGSCNYVFPYHRCI...,0.999768
1093,GIFSKLAGKAIKNLFIKGAKNVGKEVGMDVVRTGIDVIGCKIKGEC,0.999767
3615,QGVNRASCRNRWKICGAVCPRTMRQIGTCFGPRVKCCLPRWR,0.999729
3291,QKLCERSSGTWSGVCGNNNACKNQCINLEGARHGSCNYVFPAHKCI...,0.999705


In [31]:
print(results_df.shape)

(4436, 2)


In [32]:
amp_positive = (
    results_df["AMP_Probability"] > 0.5
).sum()

print("AMP Positive:", amp_positive)

print("Total:", len(filtered_df))

print(
    "Ratio:",
    amp_positive / len(filtered_df)
)

AMP Positive: 3295
Total: 4436
Ratio: 0.7427862939585211


In [ ]:
import pandas as pd

amp_df = pd.read_csv(
    "../../data/processed_data&code/clean_amp_dataset.csv"
)

train_set = set(
    amp_df["Sequence"]
)

generated_set = set(
    all_peptides
)

novel = [
    p
    for p in generated_set
    if p not in train_set
]

print(
    "Novel:",
    len(novel)
)

print(
    "Generated:",
    len(generated_set)
)

print(
    "Novelity Ratio:",
    len(novel) /
    len(generated_set)
)

Novel: 4773
Generated: 4861
Novelity Ratio: 0.981896729068093


In [34]:
import pandas as pd

generated_df = pd.read_csv(
    "long_peptides_filtered_41_60.csv"
)

print(generated_df.shape)

sequences = generated_df[
    "Sequence"
].tolist()

print(len(sequences))

(4436, 2)
4436


In [35]:
import torch
import esm

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

batch_converter = (
    alphabet.get_batch_converter()
)

model.eval()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

In [36]:
data = [
    (str(i), seq)
    for i, seq
    in enumerate(sequences)
]

In [37]:
embeddings = []

for i in range(
    0,
    len(data),
    16
):

    batch = data[
        i:i+16
    ]

    labels, strs, tokens = (
        batch_converter(batch)
    )

    tokens = tokens.to(device)

    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[33]
        )

    reps = results[
        "representations"
    ][33]

    for j, seq in enumerate(strs):

        emb = reps[
            j,
            1:len(seq)+1
        ].mean(0)

        embeddings.append(
            emb.cpu().numpy()
        )

In [38]:
import numpy as np

embeddings = np.array(
    embeddings
)

print(
    embeddings.shape
)

(4436, 1280)


In [39]:
np.save(
    "embeddings_len60.npy",
    embeddings
)

In [ ]:
import joblib

hemo_model = joblib.load(
    "../../classifers/hemolysis_xgb_classifier.pkl"
)

print(type(hemo_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [41]:
hemo_probs = hemo_model.predict_proba(
    embeddings
)[:,1]

print(hemo_probs.shape)

print(hemo_probs.min())
print(hemo_probs.max())
print(hemo_probs.mean())

(4436,)
0.0022445803
0.89819175
0.18947805


In [42]:
generated_df["Hemolysis_Probability"] = (
    hemo_probs
)

In [43]:
generated_df["AMP_Probability"] = amp_probs

In [44]:
print(hemo_probs.min())
print(hemo_probs.max())
print(hemo_probs.mean())

print((hemo_probs < 0.1).sum())
print((hemo_probs < 0.3).sum())
print((hemo_probs < 0.5).sum())

0.0022445803
0.89819175
0.18947805
1576
3562
4188


In [45]:
print(generated_df.columns)

Index(['Sequence', 'Length', 'Hemolysis_Probability', 'AMP_Probability'], dtype='object')


In [46]:
final_df = generated_df[
    (generated_df["AMP_Probability"] > 0.9)
    &
    (generated_df["Hemolysis_Probability"] < 0.3)
].copy()

print(final_df.shape)

(1749, 4)


In [47]:
final_df["Final_Score"] = (
    final_df["AMP_Probability"]
    -
    final_df["Hemolysis_Probability"]
)

final_df = final_df.sort_values(
    "Final_Score",
    ascending=False
)

final_df.head(20)

,Sequence,Length,Hemolysis_Probability,AMP_Probability,Final_Score
2263,QDRPKKPGLCPPRPQKPPCVRECKNDWSCPGEQKCCRYGCIFECRD...,51,0.003347,0.995088,0.991741
2297,QDRPKKPGLCPPRPQKPPCVRECKNDWSCPGEQQCCRYGCIYECRD...,51,0.003896,0.991691,0.987795
4000,QDRPKKGLLCPPRPQKPPCVRECKNDWRCPGEQKCCRYGCIYECRD...,52,0.003883,0.991611,0.987728
1367,GIFSKFAGKKIKNLFIKGAKNIGKEVGMDVIRTGIDVAGCKIKGEC,46,0.014036,0.999687,0.985651
1894,GISFGIKNGAISKLFGKIAKNVGKEVGMDVIRTGIDVAGCKIKGEC,46,0.014080,0.998934,0.984855
466,KDRPKKPGLCPPRPQKPCVRECKNDWSCPGEQKCCRYGCIFECRDP...,50,0.003014,0.987547,0.984533
2441,KSYGNGVYCNSKKCWVNWGQAATGMDIVTNGVTGLGGLGGAFGRPVH,47,0.013752,0.998170,0.984418
4392,KGRGSKCWNWKGVCHRDCLKTSVESGYCTRNGKLCCVKPAWHS,43,0.014293,0.998478,0.984185
2468,SKEKCWTDGKCHKVCRDTEHYVIGCNNGRFCCKRIKEQLLNPKIR,45,0.014789,0.998809,0.984021
2997,QDRPKKPGLLCPPRPQKPPCVRECKNDWSCPGEQKCCRYGCIYECR...,52,0.003871,0.987715,0.983844


In [48]:
!pip install biopython

Defaulting to user installation because normal site-packages is not writeable


In [50]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

mw_list = []
pi_list = []
arom_list = []
instab_list = []

for seq in final_df["Sequence"]:

    analysis = ProteinAnalysis(seq)

    mw_list.append(
        analysis.molecular_weight()
    )

    pi_list.append(
        analysis.isoelectric_point()
    )

    arom_list.append(
        analysis.aromaticity()
    )

    instab_list.append(
        analysis.instability_index()
    )

final_df["MolecularWeight"] = mw_list
final_df["pI"] = pi_list
final_df["Aromaticity"] = arom_list
final_df["InstabilityIndex"] = instab_list

final_df.head()

,Sequence,Length,Hemolysis_Probability,AMP_Probability,Final_Score,MolecularWeight,pI,Aromaticity,InstabilityIndex
2263,QDRPKKPGLCPPRPQKPPCVRECKNDWSCPGEQKCCRYGCIFECRD...,51,0.003347,0.995088,0.991741,5937.9487,8.896137,0.078431,53.358824
2297,QDRPKKPGLCPPRPQKPPCVRECKNDWSCPGEQQCCRYGCIYECRD...,51,0.003896,0.991691,0.987795,5953.9050,8.710468,0.078431,54.178431
4000,QDRPKKGLLCPPRPQKPPCVRECKNDWRCPGEQKCCRYGCIYECRD...,52,0.003883,0.991611,0.987728,6138.2299,9.047445,0.076923,44.623077
1367,GIFSKFAGKKIKNLFIKGAKNIGKEVGMDVIRTGIDVAGCKIKGEC,46,0.014036,0.999687,0.985651,4883.8420,9.761561,0.065217,1.408696
1894,GISFGIKNGAISKLFGKIAKNVGKEVGMDVIRTGIDVAGCKIKGEC,46,0.014080,0.998934,0.984855,4723.5832,9.448116,0.043478,-13.256522


In [51]:
print("Mean Length:",
      final_df["Length"].mean())

print("Mean MW:",
      final_df["MolecularWeight"].mean())

print("Mean pI:",
      final_df["pI"].mean())

print("Mean Aromaticity:",
      final_df["Aromaticity"].mean())

print("Mean Instability:",
      final_df["InstabilityIndex"].mean())

Mean Length: 47.71412235563179
Mean MW: 5291.9312728416235
Mean pI: 8.783086409331593
Mean Aromaticity: 0.0779621851153414
Mean Instability: 39.69885775280896


In [53]:
final_df.shape

(1749, 9)

In [54]:
final_df.to_csv(
    "final_long_amp_candidates.csv",
    index=False
)